In [3]:
import numpy as np

class PortfolioGWO:
    def __init__(self, obj_func, dim, min_weight=0.0, max_weight=0.5, agents_no=30, max_iter=100):
        self.obj_func = obj_func
        self.dim = dim
        self.lb = min_weight
        self.ub = max_weight
        self.agents_no = agents_no
        self.max_iter = max_iter

    def optimize(self):
        alpha_pos = np.zeros(self.dim)
        alpha_score = float("inf")
        beta_pos = np.zeros(self.dim)
        beta_score = float("inf")
        delta_pos = np.zeros(self.dim)
        delta_score = float("inf")

        Positions = np.random.uniform(self.lb, self.ub, (self.agents_no, self.dim))
        Positions = Positions / np.sum(Positions, axis=1, keepdims=True)

        for l in range(self.max_iter):
            for i in range(self.agents_no):
                Positions[i, :] = np.clip(Positions[i, :], self.lb, self.ub)
                if np.sum(Positions[i, :]) > 0:
                    Positions[i, :] = Positions[i, :] / np.sum(Positions[i, :])

                fitness = self.obj_func(Positions[i, :])

                if fitness < alpha_score:
                    delta_score, delta_pos = beta_score, beta_pos.copy()
                    beta_score, beta_pos = alpha_score, alpha_pos.copy()
                    alpha_score, alpha_pos = fitness, Positions[i, :].copy()
                elif alpha_score < fitness < beta_score:
                    delta_score, delta_pos = beta_score, beta_pos.copy()
                    beta_score, beta_pos = fitness, Positions[i, :].copy()
                elif beta_score < fitness < delta_score:
                    delta_score, delta_pos = fitness, Positions[i, :].copy()

            a = 2.0 - l * (2.0 / self.max_iter)

            for i in range(self.agents_no):
                for j in range(self.dim):
                    r1, r2 = np.random.rand(), np.random.rand()
                    X1 = alpha_pos[j] - (2 * a * r1 - a) * abs((2 * r2) * alpha_pos[j] - Positions[i, j])

                    r1, r2 = np.random.rand(), np.random.rand()
                    X2 = beta_pos[j] - (2 * a * r1 - a) * abs((2 * r2) * beta_pos[j] - Positions[i, j])

                    r1, r2 = np.random.rand(), np.random.rand()
                    X3 = delta_pos[j] - (2 * a * r1 - a) * abs((2 * r2) * delta_pos[j] - Positions[i, j])

                    Positions[i, j] = (X1 + X2 + X3) / 3.0

            Positions = np.clip(Positions, self.lb, self.ub)
            row_sums = np.sum(Positions, axis=1, keepdims=True)
            row_sums[row_sums == 0] = 1
            Positions = Positions / row_sums

        return alpha_pos

# === PRE-CONFIGURED DATASET ===
expected_returns = np.array([0.12, 0.10, 0.15])
cov_matrix = np.array([
    [0.005, -0.001, 0.002],
    [-0.001, 0.004, 0.001],
    [0.002, 0.001, 0.008]
])
target_return = 0.11

def risk_return_objective(weights):
    portfolio_var = np.dot(weights.T, np.dot(cov_matrix, weights))
    portfolio_ret = np.sum(weights * expected_returns)
    return_penalty = 1e4 * max(0, target_return - portfolio_ret) ** 2
    return portfolio_var + return_penalty

# === SINGLE USER INPUT QUERY ===
print("=== QUICK PORTFOLIO WEIGHT OPTIMIZER ===")
budget = float(input("Enter your total investment budget in ₹ (e.g., 50000): "))

# Execute Optimization
optimizer = PortfolioGWO(risk_return_objective, dim=3, min_weight=0.0, max_weight=0.5)
optimal_weights = optimizer.optimize()
optimal_weights = optimal_weights / np.sum(optimal_weights)

# Display Results
print("\n=== OPTIMIZED ASSET ALLOCATION ===")
for i, w in enumerate(optimal_weights):
    allocated_cash = w * budget
    print(f"Asset {i+1}: Weight = {w:.2%}, Cash Allocation = ₹{allocated_cash:,.2f}")

final_ret = np.sum(optimal_weights * expected_returns)
final_risk = np.sqrt(np.dot(optimal_weights.T, np.dot(cov_matrix, optimal_weights)))
print(f"\nExpected Portfolio Return: {final_ret:.2%}")
print(f"Portfolio Risk (Volatility): {final_risk:.2%}")


=== QUICK PORTFOLIO WEIGHT OPTIMIZER ===
Enter your total investment budget in ₹ (e.g., 50000): 250000

=== OPTIMIZED ASSET ALLOCATION ===
Asset 1: Weight = 43.19%, Cash Allocation = ₹107,975.89
Asset 2: Weight = 52.70%, Cash Allocation = ₹131,755.20
Asset 3: Weight = 4.11%, Cash Allocation = ₹10,268.91

Expected Portfolio Return: 11.07%
Portfolio Risk (Volatility): 4.14%
